# 第6章 画像処理とコンピュータビジョン ― サンプルコード
『AI実装検定 A級・S級 完全版』の第6章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、PyTorch 2.14.1、OpenCV 5.0.0.93、scikit-image 0.26.0、torchvision 0.29.1、Ultralytics 8.4.184、segment-anything 1.0

In [ ]:
!pip install -q ultralytics segment-anything   # この章で使う追加ライブラリ（必須）
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 torch==2.14.1 opencv-python==5.0.0.93 scikit-image==0.26.0 torchvision==0.29.1 ultralytics==8.4.184 segment-anything==1.0
import sys; print(sys.version)


## リスト6.1　RGB と BGR、HSV、グレースケール

In [ ]:
import numpy as np
import cv2

rgb = np.array([[[255, 0, 0], [0, 255, 0], [0, 0, 255]]], dtype=np.uint8)   # 赤・緑・青（RGB 順）
print(rgb.shape)                                      # (高さ, 幅, チャンネル)
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)            # OpenCV は B, G, R の順
print(bgr[0, 0], bgr[0, 2])                           # 赤の画素 / 青の画素
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
print(hsv[0].tolist())                                # H は 0〜179（度の半分）
print(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)[0].tolist())
print(0.299 * 255, 0.587 * 255, 0.114 * 255)

**本書に載っている出力**
```
(1, 3, 3)
[  0   0 255] [255   0   0]
[[0, 255, 255], [60, 255, 255], [120, 255, 255]]
[76, 150, 29]
76.24499999999999 149.685 29.07
```

## リスト6.2　resize の引数の順序と、補間方法の違い

In [ ]:
import numpy as np
import cv2

img = np.zeros((4, 6, 3), dtype=np.uint8)              # 高さ 4、幅 6
print(img.shape)
out = cv2.resize(img, (3, 2))                          # (幅, 高さ) の順！
print(out.shape)
a = np.array([[0, 100]], dtype=np.uint8)
print(cv2.resize(a, (4, 1), interpolation=cv2.INTER_NEAREST).tolist())
print(cv2.resize(a, (4, 1), interpolation=cv2.INTER_LINEAR).tolist())

**本書に載っている出力**
```
(4, 6, 3)
(2, 3, 3)
[[0, 0, 100, 100]]
[[0, 25, 75, 100]]
```

## リスト6.3　平均・ガウシアン・メディアンフィルタ（中心 (2,2) と角に近い (1,1) の出力）

In [ ]:
import numpy as np
import cv2

img = np.array([[10, 10, 10, 10, 10],
                [10, 10, 10, 10, 10],
                [10, 10, 90, 10, 10],
                [10, 10, 10, 10, 10],
                [10, 10, 10, 10, 10]], dtype=np.float32)   # 中央にノイズ（90）が 1 点
mean = cv2.blur(img, (3, 3))
g = np.outer([1, 2, 1], [1, 2, 1]) / 16.0                  # 3x3 のガウシアン（近似）
gauss = cv2.filter2D(img, -1, g.astype(np.float32))
med = cv2.medianBlur(img.astype(np.uint8), 3)
print(mean[2, 2], mean[1, 1])
print(gauss[2, 2], gauss[1, 1])
print(med[2, 2], med[1, 1])

**本書に載っている出力**
```
18.88889 18.88889
30.0 15.0
10 10
```

## リスト6.4　Sobel・Laplacian（縦エッジの画像、3 行目）

In [ ]:
import numpy as np
import cv2

img = np.zeros((6, 6), dtype=np.float32)
img[:, 3:] = 100                                       # 左が 0、右が 100 の縦エッジ
sx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)         # x 方向の微分
sy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)         # y 方向の微分
print(sx[2].tolist())
print(sy[2].tolist())
lap = cv2.Laplacian(img, cv2.CV_32F)
print(lap[2].tolist())
mag = np.sqrt(sx ** 2 + sy ** 2)
print(mag[2].tolist())

**本書に載っている出力**
```
[0.0, 0.0, 400.0, 400.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 100.0, -100.0, 0.0, 0.0]
[0.0, 0.0, 400.0, 400.0, 0.0, 0.0]
```

## リスト6.5　Canny エッジ検出

In [ ]:
import numpy as np
import cv2

img = np.zeros((40, 40), dtype=np.uint8)
img[10:30, 10:30] = 200                                # 明るい四角
edges = cv2.Canny(img, 50, 150)
print(np.unique(edges).tolist())                       # 0 と 255 の二値
ys, xs = np.nonzero(edges)
print(ys.min(), ys.max(), xs.min(), xs.max())          # 四角の輪郭に沿って出る
print(edges[20, 5], edges[20, 20])                     # 背景の内側・四角の中央は 0

**本書に載っている出力**
```
[0, 255]
9 29 9 29
0 0
```

## リスト6.6　大津の二値化と、モルフォロジー演算

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(0)
img = np.concatenate([rng.normal(60, 5, 500), rng.normal(180, 5, 500)])
img = np.clip(img, 0, 255).astype(np.uint8).reshape(25, 40)
t, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(t, binary.mean() / 255)                          # 2 つの山の間のしきい値 / 白の割合

m = np.zeros((7, 7), dtype=np.uint8)
m[2:5, 2:5] = 1                                        # 3x3 の白い四角
m[0, 0] = 1                                            # 孤立ノイズ
k = np.ones((3, 3), np.uint8)
ero = cv2.erode(m, k)
dil = cv2.dilate(m, k)
opened = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
print(m.sum(), ero.sum(), dil.sum(), opened.sum())

**本書に載っている出力**
```
75.0 0.5
10 1 28 9
```

## リスト6.7　64×128 画像の HOG 特徴量の次元

In [ ]:
import numpy as np
from skimage.feature import hog

img = np.random.default_rng(0).random((128, 64))       # 128 x 64 の画像
feat = hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
print(feat.shape)
cells = (128 // 8, 64 // 8)
blocks = (cells[0] - 1, cells[1] - 1)
print(cells, blocks, blocks[0] * blocks[1] * 2 * 2 * 9)

**本書に載っている出力**
```
(3780,)
(16, 8) (15, 7) 3780
```

## リスト6.8　回転行列・アフィン変換・射影変換

In [ ]:
import numpy as np
import cv2

M = cv2.getRotationMatrix2D((0, 0), 90, 1.0)           # 中心 (0, 0)、90 度（反時計回り）
print(M.round(3).tolist())
pt = np.array([1, 0, 1])                               # 点 (1, 0) を同次座標で
print((M @ pt).round(3).tolist())                      # (0, -1) 画像座標は y が下向き

src = np.float32([[0, 0], [1, 0], [0, 1]])
dst = np.float32([[0, 0], [2, 0], [0, 3]])             # x を 2 倍、y を 3 倍
A = cv2.getAffineTransform(src, dst)
print(A.tolist())

P = cv2.getPerspectiveTransform(np.float32([[0, 0], [4, 0], [4, 4], [0, 4]]),
                                np.float32([[0, 0], [4, 0], [3, 4], [1, 4]]))
v = P @ np.array([2.0, 4.0, 1.0])                      # 点 (2, 4)
print(v.round(3).tolist(), (v[:2] / v[2]).round(3).tolist())   # 最後に w で割る

**本書に載っている出力**
```
[[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0]]
[0.0, -1.0]
[[2.0, 0.0, 0.0], [0.0, 3.0, 0.0]]
[4.0, 8.0, 2.0] [2.0, 4.0]
```

## リスト6.9　IoU と NMS をゼロから実装

In [ ]:
import numpy as np

def iou(a, b):                                         # [x1, y1, x2, y2]
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return inter / (area(a) + area(b) - inter)

print(round(iou([0, 0, 4, 4], [2, 2, 6, 6]), 4))       # 重なり 2x2、和集合 28

def nms(boxes, scores, thr):
    order = np.argsort(scores)[::-1]                   # スコアの高い順
    keep = []
    while len(order):
        i = order[0]
        keep.append(int(i))
        order = np.array([j for j in order[1:] if iou(boxes[i], boxes[j]) <= thr], dtype=int)
    return keep

boxes = [[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [0, 0, 10, 9]]
scores = [0.9, 0.8, 0.7, 0.6]
print([round(iou(boxes[0], boxes[j]), 3) for j in (1, 2, 3)])
print(nms(boxes, scores, 0.5))

**本書に載っている出力**
```
0.1429
[0.681, 0.0, 0.9]
[0, 2]
```

## リスト6.10　セグメンテーションの IoU・Dice・画素の正解率

In [ ]:
import numpy as np

pred = np.array([[1, 1, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0]])
gt = np.array([[0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]])
inter = (pred & gt).sum()
union = (pred | gt).sum()
print(inter, union, round(inter / union, 4))           # IoU
print(round(2 * inter / (pred.sum() + gt.sum()), 4))   # Dice
print((pred == gt).mean())                             # 画素の正解率（背景が多いと高く出る）

**本書に載っている出力**
```
2 6 0.3333
0.5
0.6666666666666666
```

## リスト6.11　YOLO v1：1 マスの予測を画像上の箱に直す／座標の損失

In [ ]:
import numpy as np

# YOLO v1：7x7 の格子、1 マスあたり箱 2 個、20 クラス
S, B, C, img = 7, 2, 20, 448
print("出力の大きさ", S * S * (B * 5 + C))   # 7 x 7 x 30 = 1470

# --- 1 マス（行 3、列 4）の予測を、画像上の箱に直す ---
row, col, cell = 3, 4, img / S               # 1 マスは 64 画素
# x, y：マス内の相対位置  w, h：画像に対する比
# conf：物体らしさ
x, y, w, h, conf = 0.5, 0.25, 0.3, 0.6, 0.8
cx, cy = (col + x) * cell, (row + y) * cell
bw, bh = w * img, h * img
box = [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]
print("中心", cx, cy, " 幅高", round(bw, 1), round(bh, 1))
print("箱", [round(v, 1) for v in box])
# クラス別スコア = conf x P(クラス | 物体) = 0.8 x 0.9
print("クラス別スコア", round(conf * 0.9, 2))

# --- 損失（座標の項）：幅と高さは平方根をとってから差を取る ---
lam_coord = 5
gx, gy, gw, gh = 0.6, 0.2, 0.25, 0.5          # 正解
xy = (x - gx) ** 2 + (y - gy) ** 2
wh = (np.sqrt(w) - np.sqrt(gw)) ** 2 + (np.sqrt(h) - np.sqrt(gh)) ** 2
loss = lam_coord * (xy + wh)
print("xy", round(xy, 4), " wh", round(wh, 4), " 座標損失", round(loss, 4))

**本書に載っている出力**
```
出力の大きさ 1470
中心 288.0 208.0  幅高 134.4 268.8
箱 [220.8, 73.6, 355.2, 342.4]
クラス別スコア 0.72
xy 0.0125  wh 0.0068  座標損失 0.0967
```

## リスト6.12　YOLO v3 の箱の数と、IoU を距離にした k-means によるアンカー決定

In [ ]:
import numpy as np

# YOLO v3：入力 416、ストライド 32・16・8 の 3 つの縮尺で予測
size, strides, n_anchor, n_class = 416, (32, 16, 8), 3, 80
grids = [size // s for s in strides]
print("格子", grids)                                  # [13, 26, 52]
n_box = sum(n_anchor * g * g for g in grids)
print("候補の箱", n_box)                              # 3 x (169 + 676 + 2704)
print("出力チャンネル", n_anchor * (4 + 1 + n_class))  # 3 x 85

# --- アンカーを k-means で決める：距離は 1 - IoU ---
# 幅と高さだけで IoU を測る（同じ中心に置いたと考える）
def wh_iou(a, b):                      # a: (n,2)  b: (k,2)
    wi = np.minimum(a[:, None, 0], b[None, :, 0])
    hi = np.minimum(a[:, None, 1], b[None, :, 1])
    inter = wi * hi
    area_a = (a[:, 0] * a[:, 1])[:, None]
    area_b = (b[:, 0] * b[:, 1])[None, :]
    return inter / (area_a + area_b - inter)

boxes = np.array([[10, 12], [12, 10], [11, 11],
                  [30, 60], [28, 64], [32, 58],
                  [100, 90], [96, 100], [104, 94]], float)
cent = boxes[[0, 3, 6]].copy()         # 初期の中心
for _ in range(10):
    # IoU が最大 = 距離 (1 - IoU) が最小
    lab = wh_iou(boxes, cent).argmax(axis=1)
    cent = np.array([boxes[lab == k].mean(axis=0) for k in range(3)])
print(cent.round(1).tolist())
mean_iou = wh_iou(boxes, cent).max(axis=1).mean()
print("平均 IoU", round(float(mean_iou), 3))

**本書に載っている出力**
```
格子 [13, 26, 52]
候補の箱 10647
出力チャンネル 255
[[11.0, 11.0], [30.0, 60.7], [100.0, 94.7]]
平均 IoU 0.919
```

## リスト6.13　YOLOv8（軽量版 n）を設計図から組み立てる（重みは使わない）

In [ ]:
import os
os.environ["YOLO_OFFLINE"] = "1"
import torch
from ultralytics import YOLO

# 設計図（yaml）から作る。重みはダウンロードしない
model = YOLO("yolov8n.yaml")
net = model.model.eval()
# 軽量版（n）のパラメータ数
print(sum(p.numel() for p in net.parameters()))
with torch.no_grad():
    out = net(torch.zeros(1, 3, 640, 640))[0]
print(tuple(out.shape))               # (1, 4 + 80, 8400)
# 8400 = 3 つの縮尺の格子の数の合計
print(80 ** 2 + 40 ** 2 + 20 ** 2)

**本書に載っている出力**
```
3157200
(1, 84, 8400)
8400
```

## リスト6.14　SSD300 のデフォルトボックスの数と大きさ

In [ ]:
import numpy as np

# SSD300：6 つの特徴マップの大きさと、1 点あたりの箱の数
fmaps = [38, 19, 10, 5, 3, 1]
nbox = [4, 6, 6, 6, 4, 4]
total = sum(f * f * k for f, k in zip(fmaps, nbox))
print("全部で", total)                                # 8732

# 論文の式：s_k = s_min + (s_max - s_min) / (m - 1) * (k - 1)
m, s_min, s_max = 6, 0.2, 0.9
s = [s_min + (s_max - s_min) / (m - 1) * k for k in range(m)]
s.append(1.0)                                         # 最後の補助
print([round(v, 2) for v in s[:m]])

# 縦横比 a の箱：幅 = s * sqrt(a)、高さ = s / sqrt(a)
# a = 1 のときだけ、sqrt(s_k * s_k+1) の正方形をもう 1 個足す
def boxes_at(k, ratios):
    out = [(s[k] * a ** 0.5, s[k] / a ** 0.5) for a in ratios]
    out.append(((s[k] * s[k + 1]) ** 0.5,) * 2)
    return out

for k, r in ((0, [1, 2, 0.5]), (1, [1, 2, 3, 0.5, 1 / 3])):
    b = boxes_at(k, r)
    print(k, len(b), [tuple(round(v, 3) for v in x) for x in b][:3])

**本書に載っている出力**
```
全部で 8732
[0.2, 0.34, 0.48, 0.62, 0.76, 0.9]
0 4 [(0.2, 0.2), (0.283, 0.141), (0.141, 0.283)]
1 6 [(0.34, 0.34), (0.481, 0.24), (0.589, 0.196)]
```

## リスト6.15　SSD のマッチング、ハード・ネガティブ・マイニング、Smooth L1

In [ ]:
import numpy as np

def iou(a, b):                  # a: (n,4)  b: (m,4)  [x1, y1, x2, y2]
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(axis=2)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)

gt = np.array([[2, 2, 6, 6]], float)            # 正解の箱が 1 個
# デフォルトボックス 5 個
dflt = np.array([[2, 2, 6, 5], [3, 3, 7, 7], [0, 0, 3, 3],
                 [6, 6, 9, 9], [1, 5, 4, 8]], float)
v = iou(dflt, gt)[:, 0]
print(v.round(3).tolist())
pos = v >= 0.5                  # 1) IoU が 0.5 以上なら正例
pos[v.argmax()] = True          # 2) 一番重なる箱は必ず正例にする
print("正例", np.where(pos)[0].tolist())

# ハード・ネガティブ・マイニング
# 負例は「背景なのに自信がある」順に、正例の 3 倍まで使う
bg_loss = np.array([0.1, 0.2, 2.3, 0.9, 1.5])   # 各箱の背景の損失
neg_idx = np.where(~pos)[0]
k = 3 * pos.sum()
use = neg_idx[np.argsort(-bg_loss[neg_idx])][:k]
print("負例", sorted(use.tolist()))

def smooth_l1(d):               # |d| < 1：0.5 d^2、他：|d| - 0.5
    d = np.abs(d)
    return np.where(d < 1, 0.5 * d ** 2, d - 0.5)
print(smooth_l1(np.array([0.5, 2.0])).tolist())

**本書に載っている出力**
```
[0.75, 0.391, 0.042, 0.0, 0.087]
正例 [0]
負例 [2, 3, 4]
[0.125, 1.5]
```

## リスト6.16　torchvision の SSD300（VGG-16）：予測ヘッドの構造

In [ ]:
import torch
from torchvision.models.detection import ssd300_vgg16

# 学習済みの重みは使わず、構造だけ作る
net = ssd300_vgg16(weights=None, weights_backbone=None)
print(sum(p.numel() for p in net.parameters()))
# 予測ヘッド（6 つの特徴マップごとの畳み込み）
cls_head = net.head.classification_head.module_list
reg_head = net.head.regression_head.module_list
print([c.out_channels for c in cls_head])    # 箱数 x 91 クラス
print([c.out_channels for c in reg_head])    # 箱数 x 4
print([c.in_channels for c in reg_head])     # 入力チャンネル

**本書に載っている出力**
```
35641826
[364, 546, 546, 546, 364, 364]
[16, 24, 24, 24, 16, 16]
[512, 1024, 512, 256, 256, 256]
```

## リスト6.17　SAM（ViT-B）を設計図から組み立て、各部の大きさを確かめる（重みは使わない）

In [ ]:
import torch
from segment_anything import sam_model_registry

# 重み（checkpoint）なしで構造だけ作る
sam = sam_model_registry["vit_b"]()
count = lambda m: sum(p.numel() for p in m.parameters())
print("画像エンコーダ", count(sam.image_encoder))
print("プロンプトエンコーダ", count(sam.prompt_encoder))
print("マスクデコーダ", count(sam.mask_decoder))

with torch.no_grad():
    emb = sam.image_encoder(torch.zeros(1, 3, 1024, 1024))
    # 1024 / 16 = 64 → 64 x 64 マス、256 次元
    print("画像の埋め込み", tuple(emb.shape))
    pts = torch.tensor([[[500.0, 375.0]]])   # クリック 1 点（x, y）
    lab = torch.tensor([[1]])                # 1 = 前景、0 = 背景
    pe = sam.prompt_encoder
    sp, de = pe(points=(pts, lab), boxes=None, masks=None)
    print("疎なプロンプト", tuple(sp.shape))
    print("密なプロンプト", tuple(de.shape))
    low, iou = sam.mask_decoder(
        image_embeddings=emb, image_pe=pe.get_dense_pe(),
        sparse_prompt_embeddings=sp, dense_prompt_embeddings=de,
        multimask_output=True)
    print("マスク", tuple(low.shape), " 予測 IoU", tuple(iou.shape))

**本書に載っている出力**
```
画像エンコーダ 89670912
プロンプトエンコーダ 6220
マスクデコーダ 4058340
画像の埋め込み (1, 256, 64, 64)
疎なプロンプト (1, 2, 256)
密なプロンプト (1, 256, 64, 64)
マスク (1, 3, 256, 256)  予測 IoU (1, 3)
```

## リスト6.18　画像エンコーダの注意の計算量：全体の注意と窓（14×14）の注意、候補マスクの選び方

In [ ]:
import numpy as np

# 画像エンコーダ（ViT）の注意の計算量
# 全体で注意を取る場合と、窓（14x14）に区切る場合
tokens = (1024 // 16) ** 2          # 64 x 64 = 4096 トークン
win = 14 * 14                       # 窓の中のトークン数
glob = tokens ** 2                  # 全体の注意：トークン数の 2 乗
# 窓ごと：5 x 5 個の窓（端は 0 で埋めて 70 x 70 にする）
local = (64 // 14 + 1) ** 2 * win ** 2
print(tokens, glob, local, round(local / glob, 3))

# 複数のマスク候補から 1 つ選ぶ
# SAM は候補ごとに IoU を予測し、最大のものを採用する
pred_iou = np.array([0.62, 0.91, 0.78])   # 全体・部分・小さな部分
print("採用", int(pred_iou.argmax()))

**本書に載っている出力**
```
4096 16777216 960400 0.057
採用 1
```

## リスト6.19　ガンマ補正（γ = 2.2）

In [ ]:
import numpy as np
import cv2

v = np.array([0, 64, 128, 255], dtype=np.uint8)
g = 2.2
out = 255 * (v / 255.0) ** (1 / g)                     # ガンマ補正：g > 1 で暗い部分を持ち上げる
print(out.round(1).tolist())
lut = np.array([255 * (i / 255.0) ** (1 / g) for i in range(256)]).round().astype(np.uint8)
print(cv2.LUT(v.reshape(1, -1), lut).tolist())         # ルックアップテーブルで一括変換
print(round(255 * 0.5 ** (1 / g), 1))                  # 中間の灰色（127.5）はどこへ動くか

**本書に載っている出力**
```
[0.0, 136.0, 186.4, 255.0]
[[0, 136, 186, 255]]
186.1
```

## リスト6.20　ヒストグラム平坦化（4 段階の画素値の画像）

In [ ]:
import numpy as np
import cv2

img = np.array([[0, 0, 0, 1], [1, 1, 2, 2], [2, 2, 3, 3], [3, 3, 3, 3]], dtype=np.uint8)
hist = np.bincount(img.ravel(), minlength=4)
cdf = hist.cumsum()                                    # 累積度数
cdf_min = cdf[cdf > 0].min()
n = img.size
lut = np.round((cdf - cdf_min) / (n - cdf_min) * 255).astype(np.uint8)
print(hist.tolist(), cdf.tolist(), lut.tolist())
print(lut[img].tolist())
print(np.array_equal(lut[img], cv2.equalizeHist(img)))  # OpenCV の equalizeHist と同じ

**本書に載っている出力**
```
[3, 3, 4, 6] [3, 6, 10, 16] [0, 59, 137, 255]
[[0, 0, 0, 59], [59, 59, 137, 137], [137, 137, 255, 255], [255, 255, 255, 255]]
True
```

## リスト6.21　テンプレートマッチング

In [ ]:
import numpy as np
import cv2

img = np.zeros((6, 6), dtype=np.float32)
img[2:4, 3:5] = [[5, 9], [9, 5]]                       # 探したい模様を (行 2, 列 3) に置く
tpl = np.array([[5, 9], [9, 5]], dtype=np.float32)
res = cv2.matchTemplate(img, tpl, cv2.TM_SQDIFF)       # 差の 2 乗和：小さいほど似ている
print(res.shape)                                       # (6 - 2 + 1, 6 - 2 + 1)
_, _, min_loc, _ = cv2.minMaxLoc(res)
print(min_loc, round(float(res[2, 3]), 3))             # min_loc は (x, y) = (列, 行) の順
res2 = cv2.matchTemplate(img, tpl, cv2.TM_CCOEFF_NORMED)
print(round(float(res2.max()), 3))                     # 正規化相関は完全一致で 1

**本書に載っている出力**
```
(5, 5)
(3, 2) 0.0
1.0
```

## リスト6.22　ハフ変換による直線の検出

In [ ]:
import numpy as np
import cv2

img = np.zeros((40, 40), dtype=np.uint8)
img[20, 5:35] = 255                                    # 水平な線（y = 20）
lines = cv2.HoughLines(img, 1, np.pi / 180, 20)        # (rho の刻み, theta の刻み, 投票数のしきい値)
rho, theta = lines[0][0]
print(rho, round(float(theta), 4), round(np.pi / 2, 4))
x, y = 10, 20
print(round(x * np.cos(theta) + y * np.sin(theta), 3))  # 線上の点は rho = x cosθ + y sinθ を満たす

**本書に載っている出力**
```
20.0 1.5708 1.5708
20.0
```

## リスト6.23　最近傍とバイリニアで 2×2 を 4×4 に拡大し、バイリニアを自分で計算する

In [ ]:
import numpy as np
import cv2

img = np.array([[10, 20], [30, 50]], np.uint8)
# 2x2 を 4x4 に拡大（座標の対応：元 = (先 + 0.5) / 倍率 - 0.5）
near = cv2.resize(img, (4, 4), interpolation=cv2.INTER_NEAREST)
lin = cv2.resize(img, (4, 4), interpolation=cv2.INTER_LINEAR)
print(near.tolist())
print(lin.tolist())

# バイリニアを自分で計算：先の (1, 1) 画素
def bilinear(a, y, x):                 # y, x は元画像の座標（小数）
    y0, x0 = int(np.floor(y)), int(np.floor(x))
    y1, x1 = min(y0 + 1, a.shape[0] - 1), min(x0 + 1, a.shape[1] - 1)
    dy, dx = y - y0, x - x0
    top = a[y0, x0] * (1 - dx) + a[y0, x1] * dx
    bot = a[y1, x0] * (1 - dx) + a[y1, x1] * dx
    return top * (1 - dy) + bot * dy

y = x = (1 + 0.5) / 2 - 0.5            # = 0.25
print(y, bilinear(img.astype(float), y, x))

**本書に載っている出力**
```
[[10, 10, 20, 20], [10, 10, 20, 20], [30, 30, 50, 50], [30, 30, 50, 50]]
[[10, 12, 17, 20], [15, 18, 24, 28], [25, 29, 38, 43], [30, 35, 45, 50]]
0.25 18.125
```

## リスト6.24　縦のエッジの中心画素に、いろいろなフィルタを適用する

In [ ]:
import numpy as np
import cv2

img = np.zeros((5, 5), np.float32)
img[:, 3:] = 100                              # 縦のエッジ
k = {
    "Prewitt": [[-1, 0, 1]] * 3,
    "Sobel": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "エンボス": [[-2, -1, 0], [-1, 1, 1], [0, 1, 2]],
    "ラプラシアン": [[0, 1, 0], [1, -4, 1], [0, 1, 0]],
    "平均": np.full((3, 3), 1 / 9),
    "モーション": np.eye(3) / 3,               # 斜め方向の平均
}
for name, kern in k.items():
    kern = np.array(kern, np.float32)
    out = cv2.filter2D(img, -1, kern, borderType=cv2.BORDER_REPLICATE)
    print(name, round(float(out[2, 2]), 1))

win = img[1:4, 1:4]                           # 中心 (2, 2) の 3x3 窓
print("MAX-MIN", win.max() - win.min())

**本書に載っている出力**
```
Prewitt 300.0
Sobel 400.0
エンボス 300.0
ラプラシアン 100.0
平均 33.3
モーション 33.3
MAX-MIN 100.0
```

## リスト6.25　DFT を定義どおりに計算し、ローパスと DCT の直流成分を確かめる

In [ ]:
import numpy as np
import cv2

x = np.array([1, 2, 3, 4, 3, 2, 1, 0], float)
X = np.fft.fft(x)
print("直流成分", X[0].real)                  # 合計に等しい

# 定義どおりの DFT：X[k] = sum_n x[n] exp(-2 pi i k n / N)
n = len(x)
k = np.arange(n)
X2 = np.exp(-2j * np.pi * np.outer(k, k) / n) @ x
print(np.allclose(X, X2))

# ローパス：低い周波数（0, +1, -1）だけ残して戻す
mask = np.zeros(n)
mask[[0, 1, -1]] = 1
print(np.fft.ifft(X * mask).real.round(2).tolist())

# JPEG で使う DCT：一定値の 8x8 ブロックは直流成分だけが残る
blk = np.full((8, 8), 100, np.float32) - 128
d = cv2.dct(blk)
print(round(float(d[0, 0]), 1), bool(np.abs(d).sum() - abs(d[0, 0]) < 1e-3))

**本書に載っている出力**
```
直流成分 16.0
True
[0.79, 2.0, 3.21, 3.71, 3.21, 2.0, 0.79, 0.29]
-224.0 True
```

## リスト6.26　減色・ラベリング・Harris・細線化・NCC と ZNCC・ガボール

In [ ]:
import numpy as np
import cv2
from skimage.morphology import skeletonize

# 減色：256 段階を 4 段階（64 刻み）にし、各区間の中央の値にする
img = np.array([[10, 70, 130, 250]], np.uint8)
print(((img // 64) * 64 + 32).tolist())

# ラベリング：斜めに並んだ 3 画素。4 近傍と 8 近傍で数が変わる
b = np.eye(3, dtype=np.uint8)
print(cv2.connectedComponents(b, connectivity=4)[0],
      cv2.connectedComponents(b, connectivity=8)[0])    # 背景を含む数

# Harris コーナー：白い正方形（角が 4 つ）
sq = np.zeros((20, 20), np.float32)
sq[5:15, 5:15] = 1
r = cv2.cornerHarris(sq, 2, 3, 0.04)
ys, xs = np.where(r > 0.5 * r.max())
print(sorted({(int(y) // 10, int(x) // 10) for y, x in zip(ys, xs)}))

# 細線化：太さ 3 の横線を、幅 1 画素の線にする
bar = np.zeros((9, 15), bool)
bar[3:6, 2:13] = True
print(int(bar.sum()), int(skeletonize(bar).sum()))

# NCC と ZNCC：テンプレートに明るさ +50 を足した場合
t = np.array([1.0, 3.0, 2.0, 6.0])
s = t + 50
ncc = (t * s).sum() / np.sqrt((t ** 2).sum() * (s ** 2).sum())
tz, sz = t - t.mean(), s - s.mean()
zncc = (tz * sz).sum() / np.sqrt((tz ** 2).sum() * (sz ** 2).sum())
print(round(float(ncc), 4), round(float(zncc), 4))

# ガボールフィルタ：向き theta と波長 lambd で決まる
g = cv2.getGaborKernel((9, 9), 2.0, 0.0, 4.0, 0.5, 0.0)
print(g.shape, round(float(g[4, 4]), 1))

**本書に載っている出力**
```
[[32, 96, 160, 224]]
4 2
[(0, 0), (0, 1), (1, 0), (1, 1)]
33 10
0.8667 1.0
(9, 9) 1.0
```